In [5]:
import ee
import geemap

# 1. Initialize
ee.Initialize(project="gee-xplore")

# 2. Define the Region (Bhandara District, Maharashtra)
bhandara = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM2_NAME', 'Bhandara'))

# 3. Define the Kharif Window (Using the bounds from your successful graph)
start_date = '2023-06-15'
end_date = '2023-08-31'

# 4. Filter the S1 Collection
s1_kharif = (ee.ImageCollection('COPERNICUS/S1_GRD')
             .filterBounds(bhandara)
             .filterDate(start_date, end_date)
             .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
             .filter(ee.Filter.eq('instrumentMode', 'IW'))
             .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')))

# 5. The Map-Reduce Function
def process_s1_image(image):
    # Apply spatial smoothing to reduce speckle (Radius of 30 meters)
    smoothed = image.select('VH').focal_median(radius=30, units='meters')
    
    # Invert VH (so the minimum/water becomes the maximum value)
    vh_inv = smoothed.multiply(-1).rename('VH_inv')
    
    # Calculate Day of Year (DOY)
    doy = ee.Image.constant(image.date().getRelative('day', 'year')).rename('DOY').toInt()
    
    # Return an image with our necessary bands stacked together
    return image.addBands([vh_inv, doy])

# Apply the function to every image in the collection
processed_collection = s1_kharif.map(process_s1_image)

# 6. The Core GeoAI Logic: Quality Mosaic
# This single line finds the date of maximum water (minimum VH) for every pixel
# and creates a flat 2D map of the Day of Year
transplanting_map = processed_collection.qualityMosaic('VH_inv').select('DOY').clip(bhandara)

# 7. The Multi-Criteria Constraints Mask

# A. Optical Greenness (Our original mask)
s2_max_ndvi = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterBounds(bhandara).filterDate('2023-08-01', '2023-10-31')
               .map(lambda img: img.normalizedDifference(['B8', 'B4']))
               .max())
green_mask = s2_max_ndvi.gt(0.6)

# B. ESA WorldCover Land Cover (Strictly Cropland = 40)
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
crop_mask = worldcover.eq(40)

# C. SRTM Topography (Slope < 5 degrees)
dem = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(dem)
flat_mask = slope.lt(5)

# Combine all constraints (Must be Green AND Cropland AND Flat)
robust_mask = green_mask.And(crop_mask).And(flat_mask)

# Apply the bulletproof mask to our transplanting map
paddy_transplanting_clean = transplanting_map.updateMask(robust_mask)
print("Batch job logic compiled on Google Servers. Ready to visualize.")
abs# 8. Export the map to Google Drive as a Cloud Optimized GeoTIFF
print("Starting export task to Google Drive...")

export_task = ee.batch.Export.image.toDrive(
    image=paddy_transplanting_clean,
    description='Bhandara_Paddy_Transplanting_DOY_2023',
    folder='AgriPulse_Cache', # It will create this folder in your Drive
    region=bhandara.geometry(),
    scale=10, # 10m Sentinel resolution
    crs='EPSG:4326',
    maxPixels=1e10,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True} # CRITICAL for fast API reading
)

export_task.start()

# Run this to check the status of your export
import time
while export_task.active():
    print(f"Task Status: {export_task.status()['state']}...")
    time.sleep(15)

print("Export Complete! You can now download the .tif from Google Drive.")

Batch job logic compiled on Google Servers. Ready to visualize.
Starting export task to Google Drive...
Task Status: READY...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Status: RUNNING...
Task Statu

In [4]:
# 1. Initialize the interactive map
Map = geemap.Map()
Map.centerObject(bhandara, 10) # Zoom level 10 frames the district perfectly

# 2. Define the Visualization Parameters
# June 15th is roughly Day 166. August 31st is Day 243.
doy_vis = {
    'min': 166,
    'max': 243,
    'palette': [
        '#ffffcc', # Light Yellow -> Early Transplanting (Mid-June)
        '#a1dab4', # Light Green  -> Normal (July)
        '#41b6c4', # Teal         -> Normal (Late July)
        '#2c7fb8', # Blue         -> Late (August)
        '#253494'  # Dark Blue    -> Very Late / Delayed (Late August)
    ]
}

# 3. Add a basemap outline of the district for clean context
Map.addLayer(ee.Image().paint(bhandara, 0, 2), {'palette': 'black'}, 'Bhandara District Boundary')

# 4. Add the computed GeoAI layer
Map.addLayer(paddy_transplanting_map, doy_vis, 'Kharif Transplanting Date (DOY)')

# 5. Render the map
Map

Map(center=[21.159782011408943, 79.86899018794803], controls=(WidgetControl(options=['position', 'transparent_…